# 1 - Synthetic 1D Data Generation (The Basics)

This is the smallest synthetic system in `mltsa`, and it exists for one reason:
**to give us a clean toy problem where only a few observed features really matter**.

The intuition is simple. We imagine a one-dimensional transition coordinate that starts
near the top of a barrier and then falls toward one of two outcomes: `IN` or `OUT`.
In real data we usually do not observe that clean coordinate directly. Instead, we see a
set of noisy features, and only some of them carry that latent signal.

In [ ]:
from pathlib import Path
import sys

try:
    import mltsa  # noqa: F401
except ImportError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        src_dir = parent / "src"
        if (src_dir / "mltsa").exists():
            sys.path.insert(0, str(src_dir))
            break
    import mltsa  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

for parent in (Path.cwd(), *Path.cwd().parents):
    notebooks_dir = parent / "notebooks"
    if notebooks_dir.exists():
        DATA_DIR = notebooks_dir / "_generated"
        break
else:
    DATA_DIR = Path.cwd() / "_generated"

DATA_DIR.mkdir(exist_ok=True, parents=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

## Step 1: draw the intuition first

The current lightweight generator does **not** numerically integrate a physical
double-well potential. Instead, it creates a latent coordinate that gradually separates
into two classes over time. Still, it helps to keep the classical double-well picture in
mind because it explains why this toy problem is useful.

In [ ]:
x = np.linspace(-1.8, 1.8, 400)
potential = x**4 - x**2

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, potential, lw=3, color="#2a6f97")
ax.axvline(-0.707, ls="--", color="#4c956c", label="IN basin")
ax.axvline(0.707, ls="--", color="#bc4749", label="OUT basin")
ax.axvline(0.0, ls=":", color="#666666", label="transition region")
ax.set_xlabel("reaction coordinate")
ax.set_ylabel("cartoon potential")
ax.set_title("Double-well intuition for the 1D toy system")
ax.legend(loc="upper center", ncol=3)
plt.show()

## Step 2: generate a small dataset

We keep the example intentionally small. The generator gives us:

- `X`: observed time-series features
- `y`: final class labels
- `feature_names`
- `relevant_features`
- enough metadata to save, reload, rebuild, and extend the dataset later

In [ ]:
from mltsa.synthetic import make_1d_dataset

dataset = make_1d_dataset(
    n_trajectories=120,  # Number of trajectories to generate
    n_steps=64,  # Number of time steps per trajectory
    n_features=12,  # Total observed features seen by the model
    n_relevant=3,  # Hidden features that truly carry the signal
    base_seed=1234,  # Reproducible random seed
)

print("X shape:", dataset.X.shape)
print("y shape:", dataset.y.shape)
print("Relevant feature ids:", dataset.relevant_features)
print("Relevant feature names:", dataset.relevant_feature_names)
print("Generation parameters:", dataset.generation_params)

## Step 3: compare the clean latent coordinate with the observed features

Below, **before mixing** means the hidden latent coordinate that drives the outcome.
**After mixing** means the observed feature space that a model will actually see.

In [ ]:
relevant_feature = dataset.relevant_features[0]
irrelevant_feature = next(
    index for index in range(dataset.n_features) if index not in dataset.relevant_features
)
time = np.arange(dataset.n_steps)

in_indices = np.where(dataset.y == 0)[0][:4]
out_indices = np.where(dataset.y == 1)[0][:4]

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharex=True)

for index in in_indices:
    axes[0].plot(time, dataset.latent_trajectories[index, :, 0], color="#4c956c", alpha=0.8)
    axes[1].plot(time, dataset.X[index, :, relevant_feature], color="#4c956c", alpha=0.8)
    axes[2].plot(time, dataset.X[index, :, irrelevant_feature], color="#4c956c", alpha=0.8)

for index in out_indices:
    axes[0].plot(time, dataset.latent_trajectories[index, :, 0], color="#bc4749", alpha=0.8)
    axes[1].plot(time, dataset.X[index, :, relevant_feature], color="#bc4749", alpha=0.8)
    axes[2].plot(time, dataset.X[index, :, irrelevant_feature], color="#bc4749", alpha=0.8)

axes[0].set_title("Before mixing: latent coordinate")
axes[1].set_title(f"After mixing: {dataset.feature_names[relevant_feature]}")
axes[2].set_title(f"Mostly noise: {dataset.feature_names[irrelevant_feature]}")

for axis in axes:
    axis.set_xlabel("time step")
axes[0].set_ylabel("value")
plt.tight_layout()
plt.show()

In [ ]:
from mltsa.synthetic import (
    plot_ground_truth_relevance,
    plot_relevance_over_time,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_ground_truth_relevance(dataset, ax=axes[0])
plot_relevance_over_time(dataset, ax=axes[1])
plt.tight_layout()
plt.show()

## Step 4: save the dataset to HDF5

The dataset object knows how to persist itself cleanly.

In [ ]:
dataset_path = DATA_DIR / "synthetic_1d_basics.h5"
dataset.save(dataset_path, overwrite=True)
dataset_path

## Step 5: reload it and generate more trajectories later

This is handy when you want to keep the same system definition but expand the dataset.

In [ ]:
from mltsa.synthetic import load_dataset

reloaded = load_dataset(dataset_path)
extra = reloaded.generate_more(24)
extra.append_to_file(dataset_path)

expanded = load_dataset(dataset_path)
print("Trajectories before append:", reloaded.n_trajectories)
print("New trajectories generated:", extra.n_trajectories)
print("Trajectories after append:", expanded.n_trajectories)
print("Relevant features are unchanged:", expanded.relevant_feature_names)